# SIMULATED daily water demand (drivers calibrated to REAL IMD rainfall)

This notebook builds a SIMULATED daily panel dataset (2022–2024) across 198 Bengaluru wards.
Rainfall volumes are calibrated to REAL IMD monthly totals, while spatial shocks, demand multipliers, and supply failure cascades are simulated.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fix seed for reproducible simulations
np.random.seed(42)

# 1. Load wards, network edges, and IMD rainfall dataset
wards = pd.read_csv('data/wards.csv')
edges = pd.read_csv('data/edges.csv')
rainfall_df = pd.read_csv('data/raw/Bengaluru Monthly Rainfall Data (1900-2025).csv')

print('Wards loaded:', wards.shape)
print('Edges loaded:', edges.shape)
print('Rainfall columns:')
print(rainfall_df.columns.tolist())
print('\nRainfall dataset head (recent years):')
print(rainfall_df[rainfall_df['Year'].astype(str).isin(['2022', '2023', '2024'])])

Wards loaded: (198, 9)
Edges loaded: (960, 3)
Rainfall columns:
['Year', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'June', 'July', 'Aug', 'Sept', 'Oct', 'Nov', 'Dec', 'Total', 'El NiNo (Y/N)', 'La Nina (Y/N)']

Rainfall dataset head (recent years):
     Year   Jan   Feb   Mar   Apr    May   June   July    Aug   Sept    Oct   Nov   Dec   Total El NiNo (Y/N) La Nina (Y/N)
121  2022   3.9   0.0   0.0  99.1  315.0  106.6  136.7  387.1  418.0  359.8  56.8  74.7  1957.7             N             Y
122  2023   0.0   0.0   8.4  58.9  143.7   89.3  144.1  107.0  180.2  193.3  94.6   0.7  1020.2             Y             N
123  2024   0.0   0.0   0.0  43.8  274.6   75.6  118.9  140.2  151.7  202.9  52.6  93.7  1154.0             Y             Y


In [2]:
# 2. Date range and distributing REAL monthly rainfall to daily values
dates = pd.date_range('2022-01-01', '2024-12-31', freq='D')
num_days = len(dates)
num_wards = len(wards)

month_cols = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'June',
    7: 'July', 8: 'Aug', 9: 'Sept', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

city_rainfall = np.zeros(num_days)

for y in [2022, 2023, 2024]:
    row = rainfall_df[rainfall_df['Year'].astype(str) == str(y)]
    for m in range(1, 13):
        col_name = month_cols[m]
        monthly_total = float(row[col_name].values[0])
        idx = np.where((dates.year == y) & (dates.month == m))[0]
        n_days = len(idx)
        
        if monthly_total == 0.0:
            city_rainfall[idx] = 0.0
        else:
            # ~30% rainy days in monsoon (Jun-Oct), ~10% in other months
            p_rain = 0.30 if m in [6, 7, 8, 9, 10] else 0.10
            is_rain = np.random.rand(n_days) < p_rain
            if not np.any(is_rain):
                is_rain[np.random.randint(0, n_days)] = True
            
            # Distribute monthly total across rainy days
            weights = np.random.exponential(scale=1.0, size=is_rain.sum())
            weights = weights / weights.sum()
            city_rainfall[idx[is_rain]] = weights * monthly_total

# Validation check: July 2022
jul_idx = np.where((dates.year == 2022) & (dates.month == 7))[0]
real_jul_2022 = float(rainfall_df[rainfall_df['Year'].astype(str) == '2022']['July'].values[0])
print(f'Total simulation days: {num_days} (2022-01-01 to 2024-12-31)')
print(f'Check July 2022: Daily sum = {city_rainfall[jul_idx].sum():.2f} mm | Real monthly total = {real_jul_2022:.2f} mm (Match OK)')

# Micro-climate ward rainfall factor (0.8 to 1.2 fixed per ward)
ward_rain_factors = np.random.uniform(0.8, 1.2, size=num_wards)
ward_rainfall = np.outer(city_rainfall, ward_rain_factors)  # shape (1096, 198)
print('Ward rainfall matrix shape:', ward_rainfall.shape)

Total simulation days: 1096 (2022-01-01 to 2024-12-31)
Check July 2022: Daily sum = 136.70 mm | Real monthly total = 136.70 mm (Match OK)
Ward rainfall matrix shape: (1096, 198)


In [3]:
# 3. Compute baseline demand per ward per day
# Build adjacency graph dictionary
adj = {i: [] for i in range(num_wards)}
for _, r in edges.iterrows():
    u, v = int(r['src']), int(r['dst'])
    adj[u].append(v)

# Latent AR(1) shock process with SPATIAL SMOOTHING
raw_shocks = np.zeros((num_days, num_wards))
shocks = np.zeros((num_days, num_wards))

for t in range(num_days):
    if t == 0:
        raw_shocks[t] = np.random.normal(0, 0.02, size=num_wards)
    else:
        raw_shocks[t] = 0.8 * raw_shocks[t-1] + np.random.normal(0, 0.02, size=num_wards)
    
    # Spatial smoothing: 0.6 * own + 0.4 * average of neighbours
    for i in range(num_wards):
        nbr_mean = np.mean(raw_shocks[t, adj[i]]) if adj[i] else raw_shocks[t, i]
        shocks[t, i] = 0.6 * raw_shocks[t, i] + 0.4 * nbr_mean

base_mld = wards['base_mld'].values
gw_stress = wards['groundwater_stress'].values

demand = np.zeros((num_days, num_wards))
doy = dates.day_of_year.values
is_weekend = np.isin(dates.day_of_week.values, [5, 6])  # Sat / Sun
is_dry_season = np.isin(dates.month.values, [2, 3, 4, 5])  # Feb to May

for t in range(num_days):
    # Seasonal smooth bump peaking in April-May (doy ~120)
    seasonal = 1.0 + 0.15 * np.exp(-0.5 * ((doy[t] - 120) / 40.0)**2)
    weekday = 1.05 if is_weekend[t] else 1.0
    rain_effect = 1.0 - 0.002 * np.minimum(ward_rainfall[t], 50.0)
    dry_factor = 1.0 + (0.05 * gw_stress if is_dry_season[t] else 0.0)
    noise = np.random.normal(0, 0.03 * base_mld, size=num_wards)
    
    d = (base_mld * seasonal * weekday * rain_effect * (1.0 + shocks[t]) + noise) * dry_factor
    demand[t] = np.maximum(0.0, d)  # non-negative clipping

print('Baseline demand calculated with AR(1) spatial smoothing and seasonal factors.')
print(f'Shape: {demand.shape}, Min: {demand.min():.2f} MLD')

Baseline demand calculated with AR(1) spatial smoothing and seasonal factors.
Shape: (1096, 198), Min: 3.38 MLD


In [4]:
# 4. Supply failure events and network spillover cascades
num_events = 60
supply_failure = np.zeros((num_days, num_wards), dtype=int)
multiplier = np.ones((num_days, num_wards))

# Probability weights: periphery wards and dry months have higher probability
p_ward = wards['tanker_dependence'].values / wards['tanker_dependence'].sum()
date_weights = np.where(is_dry_season, 2.0, 1.0)
p_date = date_weights / date_weights.sum()

for _ in range(num_events):
    w = np.random.choice(num_wards, p=p_ward)
    t_start = np.random.choice(num_days - 10, p=p_date[:num_days - 10] / p_date[:num_days - 10].sum())
    duration = np.random.randint(3, 11)  # 3 to 10 days
    
    for dt in range(duration):
        cur_t = t_start + dt
        if cur_t < num_days:
            supply_failure[cur_t, w] = 1
            multiplier[cur_t, w] = max(multiplier[cur_t, w], 1.3)  # 1.3x failure surge
            
            # Spillover on neighbours fading over 5 days (1.10 down to 1.02)
            if dt < 5:
                spillover = 1.10 - 0.02 * dt
                for nbr in adj[w]:
                    multiplier[cur_t, nbr] = max(multiplier[cur_t, nbr], spillover)

demand = demand * multiplier
print(f'Simulated {num_events} supply failure events.')
print('Total ward-days in supply failure:', supply_failure.sum())

Simulated 60 supply failure events.
Total ward-days in supply failure: 395


In [5]:
# 5. Build consolidated DataFrame sorted by ward_id and date
dates_repeated = np.repeat(dates, num_wards)
wards_repeated = np.tile(np.arange(num_wards), num_days)

sim = pd.DataFrame({
    'date': dates_repeated,
    'ward_id': wards_repeated,
    'rainfall': ward_rainfall.flatten(),
    'demand_mld': demand.flatten(),
    'supply_failure': supply_failure.flatten()
})

sim = sim.sort_values(by=['ward_id', 'date']).reset_index(drop=True)

print('sim.shape:', sim.shape)
print('Number of failure rows:', sim['supply_failure'].sum())
print('Null counts:\n', sim.isnull().sum())
sim.head().round(2)

,date,ward_id,rainfall,demand_mld,supply_failure
0,2022-01-01,0,0.0,12.22,0
1,2022-01-02,0,0.0,12.02,0
2,2022-01-03,0,0.0,11.46,0
3,2022-01-04,0,0.0,11.39,0
4,2022-01-05,0,0.0,11.51,0


sim.shape: (217008, 5)
Number of failure rows: 395
Null counts:
date              0
ward_id           0
rainfall          0
demand_mld        0
supply_failure    0
dtype: int64


In [6]:
# 6. Plots (SIMULATED)
# a. Total city demand per day over time
daily_total = sim.groupby('date')['demand_mld'].sum()

plt.figure(figsize=(12, 4))
plt.plot(daily_total.index, daily_total.values, color='royalblue', linewidth=1.2)
plt.title('SIMULATED Total Bengaluru Daily Water Demand (2022-2024)', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Demand (MLD)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# b. One core ward vs one periphery ward in 2023
core_ward_id = wards[wards['zone'] == 'core'].iloc[0]['ward_id']
peri_ward_id = wards[wards['zone'] == 'periphery'].iloc[0]['ward_id']

sim_2023 = sim[sim['date'].dt.year == 2023]
core_2023 = sim_2023[sim_2023['ward_id'] == core_ward_id]
peri_2023 = sim_2023[sim_2023['ward_id'] == peri_ward_id]

plt.figure(figsize=(12, 4))
plt.plot(core_2023['date'], core_2023['demand_mld'], label=f'Core (Ward {core_ward_id})', color='navy')
plt.plot(peri_2023['date'], peri_2023['demand_mld'], label=f'Periphery (Ward {peri_ward_id})', color='darkorange', alpha=0.85)
plt.title('SIMULATED Daily Water Demand: Core vs Periphery Ward (2023)', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Demand (MLD)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# c. Daily rainfall of the city over 2023
city_rain_series = pd.Series(city_rainfall, index=dates)
rain_2023 = city_rain_series[city_rain_series.index.year == 2023]

plt.figure(figsize=(12, 4))
plt.bar(rain_2023.index, rain_2023.values, color='dodgerblue', width=1.0)
plt.title('SIMULATED Daily Rainfall of Bengaluru (2023) Calibrated to IMD Totals', fontsize=12, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Rainfall (mm)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [7]:
# 7. Validation checks
print('=== VALIDATION CHECKS ===')
print(f'Mean total daily demand: {daily_total.mean():.2f} MLD (Expected near 2632 baseline + seasonal/weekend peaks)')
print(f'Minimum demand: {sim["demand_mld"].min():.2f} MLD (Check >= 0: {sim["demand_mld"].min() >= 0})')

# Spatial correlation check
w0_demand = sim[sim['ward_id'] == 0]['demand_mld'].values
nbr_id = adj[0][0]
nbr_demand = sim[sim['ward_id'] == nbr_id]['demand_mld'].values

# Find geographically most distant ward from ward 0
dists = np.sqrt((wards['x'] - wards.loc[0, 'x'])**2 + (wards['y'] - wards.loc[0, 'y'])**2)
far_id = dists.argmax()
far_demand = sim[sim['ward_id'] == far_id]['demand_mld'].values

corr_nbr = np.corrcoef(w0_demand, nbr_demand)[0, 1]
corr_far = np.corrcoef(w0_demand, far_demand)[0, 1]

print(f'Correlation with nearest neighbour (Ward {nbr_id}): {corr_nbr:.4f}')
print(f'Correlation with distant ward (Ward {far_id}): {corr_far:.4f}')
print('Higher correlation with nearest neighbour confirmed (Spatial dependence verified).')

=== VALIDATION CHECKS ===
Mean total daily demand: 2794.16 MLD (Expected near 2632 baseline + seasonal/weekend peaks)
Minimum demand: 3.38 MLD (Check >= 0: True)
Correlation with nearest neighbour (Ward 138): 0.8098
Correlation with distant ward (Ward 134): 0.7533
Higher correlation with nearest neighbour confirmed (Spatial dependence verified).


In [8]:
# 8. Save simulated demand panel dataset
os.makedirs('data', exist_ok=True)
sim.to_csv('data/simulated_demand.csv', index=False)
print(f'Saved data/simulated_demand.csv ({len(sim)} rows)')

Saved data/simulated_demand.csv (217008 rows)


## Data Provenance Note: REAL vs SIMULATED Components

- **REAL Components:**
  - **IMD Monthly Rainfall Totals:** Real historical precipitation measurements for Bengaluru (2022–2024) loaded from `data/raw/`.
  - **Baseline City-Wide Demand Target:** Calibrated to match the benchmark city water requirement of exactly **2,632 MLD**.

- **SIMULATED Components:**
  - **Daily Rain Distribution:** Synthetic spreading of monthly rainfall totals over monsoon (30%) and non-monsoon (10%) days.
  - **Micro-climate Variation:** Random 0.8–1.2 multipliers applied per ward.
  - **Latent AR(1) Shock Process:** Spatially smoothed shocks across 4-nearest-neighbour graph edges.
  - **Supply Failure Cascades:** 60 stochastic failure events with 1.3x demand surges and decaying 1.1x neighbor spillovers.
  - **Ward Demand Records:** All daily ward-level `demand_mld` values are synthetically generated.